### **Data Aggregation**

**Goal**

Integrate multiple datasets from open-source, community-driven platforms to identify duplicate data center records, enrich existing information, and fill missing values using matched data from multiple sources.

**Helper Functions**

Functions from `data_utils.py` are used to standardize addresses by removing special characters and extra whitespace and replacing common address terms with predefined abbreviations. In the crypto dataset, the `Address` field sometimes also contains the city, state, and ZIP code. A second function, `remove_city_state`, removes this duplicated location information, while ZIP codes are extracted into a separate column.

In [25]:
import re
import pandas as pd
import numpy as np
from geopy.distance import geodesic

from data_utils import (
    calculate_distance, 
    remove_city_state, 
    standardize_address
)

In [26]:
ft_data_centers = pd.read_csv('./data/raw/fractracker_data_centers.csv')
ft_crypto_centers = pd.read_csv('./data/raw/fractracker_crypto.csv')
atlas_data_centers = pd.read_csv('./data/raw/compute-atlas-facilities.csv')

print(f"FracTracker Data Centers {ft_data_centers.shape}")
print(f"FracTracker Crypto Centers {ft_crypto_centers.shape}")
print(f"Compute Atlas Data Centers {atlas_data_centers.shape}")

FracTracker Data Centers (1720, 53)
FracTracker Crypto Centers (302, 18)
Compute Atlas Data Centers (2659, 20)


**Standardize Columns**

In [27]:
ft_crypto_centers.columns = [
    re.sub(r'[?():]+', '', col.lower().replace(' ', '_'))
    for col in ft_crypto_centers.columns
]

atlas_data_centers = atlas_data_centers.rename(columns={'lon': 'long'})

In [28]:
atlas_cols = [
    'name', 'operator', 'facility_type', 'lat', 'long', 
    'capacity_operational_mw', 'capacity_planned_mw'
]

crypto_cols = [
    'date', 'name', 'address', 'city', 'state', 'lat', 'long',
    'company', 'probable_power_source', 'mw_nytimes'
]

atlas_data_centers = atlas_data_centers[atlas_cols]
ft_crypto_centers = ft_crypto_centers[crypto_cols]

### **Dataset Comparison**

The FracTracker data center list does not say which facilities are crypto mines. We build an `is_crypto` label here so `03-supervised-learning.ipynb` can load it from `data/processed/` after cleaning.

The label comes from **two different sources**:

1. **The FracTracker crypto dataset.** A data center is crypto if it matches a known crypto mining site, either:
   - by location: a crypto site is within `MAX_MILES` of it, or
   - by exact address (the Address Comparison above).
2. **The FracTracker data centers dataset itself.** Some data centers already say they are crypto in their `purpose` or `facility_name` (for example "Bitcoin" or "Crypto mining/AI"). Many of these are not in the crypto dataset at all.

We keep one column for each check (`crypto_from_distance`, `crypto_from_address`, `crypto_from_dc_dataset`) so we can always see where a label came from.

##### **How old is each dataset?**

The crypto dataset is older than the data centers dataset:

- **Crypto sites:** the `Date` column runs from March 2023 to April 2026, but most entries are from 2023 and 2024 (median: May 2024).
- **Data centers:** the `date_updated` column runs from January 2025 to September 2026 (median: November 2025).

So there is roughly a year and a half between the typical crypto entry and the typical data center entry. This matters for the crypto label:

- A data center that matched a crypto site tells us the site **was** crypto when the crypto dataset was made. It may not still be crypto. Several sites are converting from crypto mining to AI (for example Applied Digital, Crexi and Keel Infrastructure).
- A "Proposed" crypto site from 2023 may never have been built, or may have been built as a different kind of data center.
- The `crypto_from_dc_dataset` column comes from the newer dataset, so it is the more up-to-date of the two sources.

In [29]:
# Compare how old each dataset is
crypto_dates = pd.to_datetime(ft_crypto_centers["date"], format="%m/%d/%Y", errors="coerce")
dc_dates = pd.to_datetime(ft_data_centers["date_updated"], unit="ms")

print("Crypto sites with a date:", crypto_dates.notna().sum(), "of", len(ft_crypto_centers))
print("Crypto dates:            ", crypto_dates.min().date(), "to", crypto_dates.max().date())
print("Crypto median date:      ", crypto_dates.median().date())
print()
print("Data centers last updated:", dc_dates.min().date(), "to", dc_dates.max().date())
print("Data centers median date: ", dc_dates.median().date())
print()
print("Crypto sites by year:")
print(crypto_dates.dt.year.value_counts().sort_index())
print()
print("Data centers by year last updated:")
print(dc_dates.dt.year.value_counts().sort_index())

Crypto sites with a date: 167 of 302
Crypto dates:             2023-03-02 to 2026-04-15
Crypto median date:       2024-05-17

Data centers last updated: 2025-01-04 to 2026-10-20
Data centers median date:  2025-11-24

Crypto sites by year:
date
2023.0    39
2024.0    96
2025.0    30
2026.0     2
Name: count, dtype: int64

Data centers by year last updated:
date_updated
2025.0    892
2026.0    825
Name: count, dtype: int64


#### **Distance Matching**

**Distance between every crypto site and every data center**

`geodesic` from the `geopy` package takes two `(lat, long)` points and returns the distance between them along the Earth's surface.

We loop over every crypto site and every data center, which is about 500,000 pairs, so this cell takes around 30 seconds.

The result is a table with one row per crypto site and one column per data center.

In [30]:
crypto_points = list(zip(ft_crypto_centers["lat"], ft_crypto_centers["long"]))
dc_points = list(zip(ft_data_centers["lat"], ft_data_centers["long"]))

distances = []
for crypto_point in crypto_points:
    row = []
    for dc_point in dc_points:
        row.append(geodesic(crypto_point, dc_point).miles)
    distances.append(row)

# turn the list of rows into a table numpy can search
distances = np.array(distances)

print("Distance table (crypto sites, data centers):", distances.shape)

Distance table (crypto sites, data centers): (302, 1720)


**Nearest data center for each crypto site**

Before picking a cutoff, look at how far each crypto site is from its closest data center.

In [31]:
nearest_dc = distances.argmin(axis=1)

ft_crypto_centers["nearest_dc_miles"] = distances.min(axis=1)
ft_crypto_centers["nearest_dc_name"] = ft_data_centers["facility_name"].values[nearest_dc]
ft_crypto_centers["nearest_dc_operator"] = ft_data_centers["operator_name"].values[nearest_dc]

for miles in [0.075, 0.1, 0.25, 0.5, 0.75, 1, 2, 5]:
    # keep only the crypto sites that are close enough, then count them
    nearby = ft_crypto_centers[ft_crypto_centers["nearest_dc_miles"] <= miles]
    print(f"Crypto sites with a data center within {miles} mi: {len(nearby)}")

Crypto sites with a data center within 0.075 mi: 22
Crypto sites with a data center within 0.1 mi: 24
Crypto sites with a data center within 0.25 mi: 33
Crypto sites with a data center within 0.5 mi: 42
Crypto sites with a data center within 0.75 mi: 45
Crypto sites with a data center within 1 mi: 49
Crypto sites with a data center within 2 mi: 58
Crypto sites with a data center within 5 mi: 75


### **Aggregate Datasets**

Using a radius of approximately **0.075 miles** provides a good balance between identifying facilities across datasets and reducing the risk of incorrectly matching facilities that are simply located near one another.

**Address Standardization & Comparison**

Addresses are standardized across both datasets to make facility matching more reliable. Common street names, directions, and unit types are converted to consistent abbreviations, while special characters and extra whitespace are removed. The crypto dataset sometimes stores the city, state, and ZIP code inside the Address field. 

In [32]:
# Extract ZIP before removing it from the address
ft_crypto_centers["zip"] = (
    ft_crypto_centers["address"]
    .str.extract(r"(\d{5})\s*$")[0]
    .replace('', np.nan)
    .astype("Int64")
)

ft_crypto_centers["address"] = ft_crypto_centers.apply(
    lambda row: remove_city_state(
        row["address"],
        row["city"],
        row["state"]
    ),
    axis=1
)

# Standardize addresses
ft_crypto_centers["address"] = ft_crypto_centers["address"].apply(standardize_address)
ft_data_centers["address"] = ft_data_centers["address"].apply(standardize_address)

In [33]:
ft_crypto_centers["crypto_idx"] = ft_crypto_centers.index # add id field for merging
address_cols = ["address", "city", "state"]

# Exclude missing address
address_lookup = (
    ft_crypto_centers
    .dropna(subset=address_cols)[["address", "city", "state", "crypto_idx"]]
    .drop_duplicates(subset=address_cols)
)

# Find exact address matches
matched = pd.merge(
    ft_data_centers,
    address_lookup,
    on=address_cols,
    how="left"
)

address_matches = len(matched.loc[matched["crypto_idx"].notna()])

print(f"{len(matched.loc[matched["crypto_idx"].notna()])} matches found.")

20 matches found.


In [34]:
# Search for a crypto data centers within 0.075 miles
matched.loc[matched["crypto_idx"].isna(), "crypto_idx"] = calculate_distance(
    matched.loc[matched["crypto_idx"].isna()], 
    ft_crypto_centers,
    max_distance=0.075
)

merged_data_centers = pd.merge(
    matched,
    ft_crypto_centers.drop(columns="crypto_idx"),
    how="left",
    left_on="crypto_idx",
    right_index=True,
    suffixes=("", "_crypto")
)

print(f"{len(matched.loc[matched["crypto_idx"].notna()]) - address_matches} matches found.")

9 matches found.


**Compute Atlas Data Centers**

The following adds additional data points from the compute atlas dataset in order to filling in missing values found in FracTracker. This function identifies roughly **200** instances where the data overlaps.

In [35]:
# Search for a atlas data centers within 0.075 miles
merged_data_centers['atlas_idx'] = calculate_distance(
    merged_data_centers, 
    atlas_data_centers, 
    max_distance=0.075
)

merged_data_centers = pd.merge(
    merged_data_centers, 
    atlas_data_centers, 
    how='left', 
    left_on='atlas_idx',
    right_index=True,
    suffixes=("", "_atlas")
)

print(f"{len(merged_data_centers.loc[merged_data_centers['atlas_idx'].notna()])} matches found.")

197 matches found.


##### **Save Results**

In [36]:
# Save data file
merged_data_centers.to_csv('./data/processed/merged_data_centers.csv', index=False)

merged_data_centers.head()

,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,...,nearest_dc_operator,zip_crypto,atlas_idx,name_atlas,operator,facility_type,lat_atlas,long_atlas,capacity_operational_mw,capacity_planned_mw
0,3973423,3d8a6ce3-5e4b-4ede-b9ca-c9ab4f5dfa9b,1779305646902-5353,Stak Energy Data Center,dalton hwy 26 miles s of deadhorse,North Slope Borough,AK,99734.0,North Slope,69.90071,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3973424,c65854df-23a2-4d76-8ab7-c24323cb63f9,1779306105674-8821,Prudhoe Bay Data Center,dalton hwy,Prudhoe Bay,AK,99734.0,North Slope,70.18478,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3973425,8cc98adc-12ef-484e-bf6e-2ee2b94df11a,1,Project Marvel,red rdg on rock mtn lk rd and old lawnstown rd,Bessemer,AL,35022.0,Jefferson,33.34200,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3973426,d4b7f5c8-6123-433b-bae9-0bb07e997636,1771618379668-1750,BHM01 Nebius Data Center,201 milan pkwy,Birmingham,AL,35221.0,Jefferson,33.42746,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3973427,4c93a9bd-0bd3-426f-bb7b-69b0174fea3a,2,DC Blox,433 6th st s,Birmingham,AL,35233.0,Jefferson,33.50051,...,NaN,<NA>,734.0,DC BLOX Birmingham Data Center Campus,DC BLOX,data_center,33.500665,-86.820831,5.0,60.0
